In [28]:
import numpy as np
import pyaudio
import threading


class Tone:
    def __init__(self, volume=0.5, rate=48000, channels=1, chunk=1024):
        self.volume = volume
        self.rate = rate
        self.channels = channels
        self.chunk = chunk

        self.p = pyaudio.PyAudio()

        self.lock = threading.Lock()
        self.buffer = np.array([], dtype=np.float32)
        self.playing = False

        self.stream = self.p.open(
            format=pyaudio.paFloat32,
            channels=self.channels,
            rate=self.rate,
            output=True,
            frames_per_buffer=self.chunk,
            stream_callback=self._callback
        )

        self.stream.start_stream()

    def _callback(self, in_data, frame_count, time_info, status):
        with self.lock:
            need = frame_count

            if len(self.buffer) >= need:
                out = self.buffer[:need]
                self.buffer = self.buffer[need:]
            else:
                out = np.zeros(need, dtype=np.float32)
                if len(self.buffer) > 0:
                    out[:len(self.buffer)] = self.buffer
                    self.buffer = np.array([], dtype=np.float32)

            self.playing = len(self.buffer) > 0

        return (out.astype(np.float32).tobytes(), pyaudio.paContinue)

    def _freq(self, octave=3, note=1):
        # octave=3, note=10 -> A4(라) = 440Hz
        return 2 ** octave * 55 * 2 ** ((note - 10) / 12)

    def _make_tone(self, octave=3, note=1, duration=0.5):
        n = int(self.rate * duration)

        if note == 0:
            return np.zeros(n, dtype=np.float32)

        f = self._freq(octave, note)

        sample = np.sin(
            2 * np.pi * np.arange(n) * f / self.rate
        ).astype(np.float32)

        # 시작/끝 클릭음 완화
        fade_n = min(int(self.rate * 0.01), n // 2)
        if fade_n > 0:
            fade_in = np.linspace(0.0, 1.0, fade_n, dtype=np.float32)
            fade_out = np.linspace(1.0, 0.0, fade_n, dtype=np.float32)
            sample[:fade_n] *= fade_in
            sample[-fade_n:] *= fade_out

        return (self.volume * sample).astype(np.float32)

    def play(self, octave=3, note=1, duration=0.5):
        tone = self._make_tone(octave, note, duration)

        with self.lock:
            self.buffer = np.concatenate([self.buffer, tone])
            self.playing = True

    def rest(self, duration=0.2):
        silence = np.zeros(int(self.rate * duration), dtype=np.float32)

        with self.lock:
            self.buffer = np.concatenate([self.buffer, silence])
            self.playing = True

    def play_song(self, song):
        # song 형식: [(octave, note, duration), ...]
        for octave, note, duration in song:
            if note == 0:
                self.rest(duration)
            else:
                self.play(octave, note, duration)

    def play_school_bell(self):
        school_bell = [
            (3, 8, 0.40), (3, 8, 0.40), (3, 10, 0.40), (3, 10, 0.40),
            (3, 8, 0.40), (3, 8, 0.40), (3, 5, 0.80),

            (3, 8, 0.40), (3, 8, 0.40), (3, 5, 0.40), (3, 5, 0.40),
            (3, 3, 0.80),

            (3, 8, 0.40), (3, 8, 0.40), (3, 10, 0.40), (3, 10, 0.40),
            (3, 8, 0.40), (3, 8, 0.40), (3, 5, 0.80),

            (3, 8, 0.40), (3, 5, 0.40), (3, 3, 0.40), (3, 5, 0.40),
            (3, 1, 1.00),
        ]
        self.play_song(school_bell)

    def play_mission_impossible(self):
        s = 0.12
        m = 0.22
        l = 0.34
        r = 0.05

        mission = [
            (4, 8, s),  (0, 0, r),
            (4, 8, s),  (0, 0, r),
            (4, 8, s),
            (4, 11, s),
            (5, 1, m),

            (4, 8, s),  (0, 0, r),
            (4, 8, s),  (0, 0, r),
            (4, 7, s),
            (4, 8, m),

            (4, 8, s),  (0, 0, r),
            (4, 8, s),  (0, 0, r),
            (4, 8, s),
            (4, 11, s),
            (5, 1, m),

            (4, 8, s),  (0, 0, r),
            (4, 8, s),  (0, 0, r),
            (4, 7, s),
            (4, 8, l),
        ]
        self.play_song(mission)

    def play_game_start(self):
        s = 0.10
        m = 0.18
        l = 0.30
        r = 0.03

        game_start = [
            (4, 1, s), (0, 0, r),
            (4, 5, s), (0, 0, r),
            (4, 8, s), (0, 0, r),
            (5, 1, m), (0, 0, r),

            (4, 8, s), (0, 0, r),
            (5, 1, s), (0, 0, r),
            (5, 5, m), (0, 0, r),

            (5, 8, l),
        ]
        self.play_song(game_start)

    def is_playing(self):
        with self.lock:
            return self.playing or len(self.buffer) > 0

    def clear(self):
        with self.lock:
            self.buffer = np.array([], dtype=np.float32)
            self.playing = False

    def close(self):
        self.clear()
        self.stream.stop_stream()
        self.stream.close()
        self.p.terminate()

In [29]:
import time

tone = Tone(volume=0.1)
tone.play_school_bell()

for i in range(100):
    print("main work", i)
    time.sleep(0.1)

main work 0
main work 1
main work 2
main work 3
main work 4
main work 5
main work 6
main work 7
main work 8
main work 9
main work 10
main work 11
main work 12
main work 13
main work 14
main work 15
main work 16
main work 17
main work 18
main work 19
main work 20
main work 21
main work 22
main work 23
main work 24
main work 25
main work 26
main work 27
main work 28
main work 29
main work 30
main work 31
main work 32
main work 33
main work 34
main work 35
main work 36
main work 37
main work 38
main work 39
main work 40
main work 41
main work 42
main work 43
main work 44
main work 45
main work 46
main work 47
main work 48
main work 49
main work 50
main work 51
main work 52
main work 53
main work 54
main work 55
main work 56
main work 57
main work 58
main work 59
main work 60
main work 61
main work 62
main work 63
main work 64
main work 65
main work 66
main work 67
main work 68
main work 69
main work 70
main work 71
main work 72
main work 73
main work 74
main work 75
main work 76
main work

In [30]:
import time

tone = Tone(volume=0.1)
tone.play_mission_impossible()

for i in range(100):
    print("main work", i)
    time.sleep(0.1)

main work 0
main work 1
main work 2
main work 3
main work 4
main work 5
main work 6
main work 7
main work 8
main work 9
main work 10
main work 11
main work 12
main work 13
main work 14
main work 15
main work 16
main work 17
main work 18
main work 19
main work 20
main work 21
main work 22
main work 23
main work 24
main work 25
main work 26
main work 27
main work 28
main work 29
main work 30
main work 31
main work 32
main work 33
main work 34
main work 35
main work 36
main work 37
main work 38
main work 39
main work 40
main work 41
main work 42
main work 43
main work 44
main work 45
main work 46
main work 47
main work 48
main work 49
main work 50
main work 51
main work 52
main work 53
main work 54
main work 55
main work 56
main work 57
main work 58
main work 59
main work 60
main work 61
main work 62
main work 63
main work 64
main work 65
main work 66
main work 67
main work 68
main work 69
main work 70
main work 71
main work 72
main work 73
main work 74
main work 75
main work 76
main work

In [31]:
import time

tone = Tone(volume=0.1)
tone.play_game_start()

for i in range(100):
    print("main work", i)
    time.sleep(0.1)

main work 0
main work 1
main work 2
main work 3
main work 4
main work 5
main work 6
main work 7
main work 8
main work 9
main work 10
main work 11
main work 12
main work 13
main work 14
main work 15
main work 16
main work 17
main work 18
main work 19
main work 20
main work 21
main work 22
main work 23
main work 24
main work 25
main work 26
main work 27
main work 28
main work 29
main work 30
main work 31
main work 32
main work 33
main work 34
main work 35
main work 36
main work 37
main work 38
main work 39
main work 40
main work 41
main work 42
main work 43
main work 44
main work 45
main work 46
main work 47
main work 48
main work 49
main work 50
main work 51
main work 52
main work 53
main work 54
main work 55
main work 56
main work 57
main work 58
main work 59
main work 60
main work 61
main work 62
main work 63
main work 64
main work 65
main work 66
main work 67
main work 68
main work 69
main work 70
main work 71
main work 72
main work 73
main work 74
main work 75
main work 76
main work